In [ ]:
#Task 1

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

np.random.seed(42)
RANDOM_STATE = 42
VAR_SMOOTHING = 1e-9

data = load_breast_cancer()

X = data.data.astype(float)
y = data.target.astype(int)
feature_names = data.feature_names
classes = np.unique(y)

print("Dataset shape:", X.shape)
print("Classes:", data.target_names)
print("Class counts:", {int(c): int(np.sum(y == c)) for c in classes})

Dataset shape: (569, 30)
Classes: ['malignant' 'benign']
Class counts: {0: 212, 1: 357}


In [8]:
import pandas as pd

df = pd.DataFrame(X, columns=feature_names)

df["target"] = y

df["diagnosis"] = data.target_names[y]

display(df.head(10))

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0,malignant
5,12.45,15.70,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,0.07613,...,103.40,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.12440,0,malignant
6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,0.05742,...,153.20,1606.0,0.1442,0.2576,0.3784,0.1932,0.3063,0.08368,0,malignant
7,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,0.07451,...,110.60,897.0,0.1654,0.3682,0.2678,0.1556,0.3196,0.11510,0,malignant
8,13.00,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,0.07389,...,106.20,739.3,0.1703,0.5401,0.5390,0.2060,0.4378,0.10720,0,malignant
9,12.46,24.04,83.97,475.9,0.11860,0.23960,0.22730,0.08543,0.2030,0.08243,...,97.65,711.4,0.1853,1.0580,1.1050,0.2210,0.4366,0.20750,0,malignant


In [9]:

X = data.data.astype(float)
y = data.target.astype(int)
feature_names = data.feature_names
classes = np.unique(y)

print("Dataset shape:", X.shape)
print("Classes:", data.target_names)
print("Class counts:", {int(c): int(np.sum(y == c)) for c in classes})

Dataset shape: (569, 30)
Classes: ['malignant' 'benign']
Class counts: {0: 212, 1: 357}


In [10]:
def train_test_split(X, y, test_size=0.2, seed=42):
    X = np.asarray(X)
    y = np.asarray(y)
    rng = np.random.default_rng(seed)
    train_indices, test_indices = [], []

    for label in np.unique(y):
        class_indices = np.where(y == label)[0].copy()
        rng.shuffle(class_indices)
        n_test = int(np.ceil(len(class_indices) * test_size))
        test_indices.extend(class_indices[:n_test])
        train_indices.extend(class_indices[n_test:])

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    train_indices = np.asarray(train_indices, dtype=int)
    test_indices = np.asarray(test_indices, dtype=int)
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, seed=RANDOM_STATE)

print("Training class counts:", {int(c): int(np.sum(y_train == c)) for c in classes})
print("Testing class counts:", {int(c): int(np.sum(y_test == c)) for c in classes})
print("Train/test shapes:", X_train.shape, X_test.shape)


train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
train_std[train_std == 0] = 1.0
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

Training class counts: {0: 169, 1: 285}
Testing class counts: {0: 43, 1: 72}
Train/test shapes: (454, 30) (115, 30)


In [11]:
class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape
        self.theta_ = np.zeros((len(self.classes_), n_features), dtype=float)
        self.var_ = np.zeros((len(self.classes_), n_features), dtype=float)
        self.class_prior_ = np.zeros(len(self.classes_), dtype=float)

        epsilon = self.var_smoothing * np.var(X, axis=0).max()
        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.theta_[idx, :] = X_c.mean(axis=0)
            self.var_[idx, :] = X_c.var(axis=0) + epsilon
            self.class_prior_[idx] = X_c.shape[0] / n_samples
        return self

    def predict_log_proba(self, X):
        X = np.asarray(X, dtype=float)
        # Shape: (samples, classes, features)
        joint = (
            -0.5 * np.sum(np.log(2.0 * np.pi * self.var_), axis=1)
            -0.5 * np.sum(((X[:, None, :] - self.theta_[None, :, :]) ** 2)
                          / self.var_[None, :, :], axis=2)
            + np.log(self.class_prior_)[None, :]
        )
        # Normalize log probabilities using log-sum-exp.
        row_max = np.max(joint, axis=1, keepdims=True)
        log_sum_exp = row_max + np.log(np.sum(np.exp(joint - row_max), axis=1, keepdims=True))
        return joint - log_sum_exp

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        log_probs = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_probs, axis=1)]

In [12]:
X_tiny = np.array([[1.0], [3.0], [5.0], [7.0]])
y_tiny = np.array([0, 0, 1, 1])
tiny_model = GaussianNaiveBayes(var_smoothing=0.0).fit(X_tiny, y_tiny)

manual_mean_0 = np.mean([1.0, 3.0])
manual_var_0 = np.var([1.0, 3.0])  # population variance, ddof=0
x_value = 2.0
manual_likelihood_0 = (1 / np.sqrt(2 * np.pi * manual_var_0)) * np.exp(
    -((x_value - manual_mean_0) ** 2) / (2 * manual_var_0)
)

model_mean_0 = tiny_model.theta_[0, 0]
model_var_0 = tiny_model.var_[0, 0]
model_likelihood_0 = (1 / np.sqrt(2 * np.pi * model_var_0)) * np.exp(
    -((x_value - model_mean_0) ** 2) / (2 * model_var_0)
)

print("Manual class-0 mean:", manual_mean_0, "| Model:", model_mean_0)
print("Manual class-0 variance:", manual_var_0, "| Model:", model_var_0)
print("Manual Gaussian likelihood at x=2:", manual_likelihood_0)
print("Model Gaussian likelihood at x=2:", model_likelihood_0)
assert np.allclose(manual_mean_0, model_mean_0)
assert np.allclose(manual_var_0, model_var_0)
assert np.allclose(manual_likelihood_0, model_likelihood_0)
print("Sanity test passed.")

Manual class-0 mean: 2.0 | Model: 2.0
Manual class-0 variance: 1.0 | Model: 1.0
Manual Gaussian likelihood at x=2: 0.3989422804014327
Model Gaussian likelihood at x=2: 0.3989422804014327
Sanity test passed.
